In [1]:
#!/usr/bin/env python
# coding: utf-8

import math
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
nltk.download('stopwords')
from nltk.tokenize import word_tokenize
import pandas as pd
from datetime import datetime
import pickle
import re
from pyxdameraulevenshtein import damerau_levenshtein_distance
import apsw
import sys
import numpy as np
import corp_simplify_utils
import seaborn as sns
import matplotlib.pyplot as plt
import pyreadr
from collections import Counter

# nlp
import spacy
from spacy import displacy
from collections import Counter
# to install: $python3 -m spacy download en_core_web_lg
import en_core_web_lg

# analysis/regressions
import statsmodels.api as sm
from statsmodels.formula.api import glm
from statsmodels.genmod.families import Poisson
from scipy.stats import ks_2samp
from scipy.stats import mannwhitneyu
from scipy.stats import ttest_ind

# from statsmodels.graphics.gofplots import qqplot_2samples
from scipy import stats
from joypy import joyplot
from matplotlib import cm

from datetime import date
today_for_filenames = date.today()
curr_date = str(today_for_filenames.strftime("%Y%m%d"))


NUMBER_OF_MATCHES_TO_RECORD = 10
punc_remove_re = re.compile(r'\W+')
corp_re = re.compile('( (group|holding(s)?( co)?|inc(orporated)?|ltd|l ?l? ?[cp]|co(rp(oration)?|mpany)?|s[ae]|plc))+$')
and_re = re.compile(' & ')
punc1_re = re.compile(r'(?<=\S)[\'’´\.](?=\S)')
punc2_re = re.compile(r'[\s\.,:;/\'"`´‘’“”\(\)\[\]\{\}_—\-?$=!]+')

STOPWORDS = nltk.corpus.stopwords.words('english')
STOPWORDS.remove("am")
STOPWORDS.remove("up")
STOPWORDS.remove("in")
STOPWORDS.remove("on")
STOPWORDS.remove("all")
STOPWORDS.remove("any")
STOPWORDS.remove("most")
STOPWORDS.remove("no")
STOPWORDS.remove("nor")
STOPWORDS.remove("own")
STOPWORDS.remove("same")
STOPWORDS.remove("so")
STOPWORDS.remove("very")
STOPWORDS.remove("s")
STOPWORDS.remove("t")
STOPWORDS.remove("d")
STOPWORDS.remove("ll")
STOPWORDS.remove("m")
STOPWORDS.remove("o")
STOPWORDS.remove("re")
STOPWORDS.remove("ve")
STOPWORDS.remove("y")

#compile regex patterns to reuse
STOPWORD_RE = re.compile(r'\b(the|of|and|in|on)\b', re.IGNORECASE)
CORP_SUFFIX_RE = re.compile(r'\b(inc|corp|ltd|llc|plc|co|company|limited)\b', re.IGNORECASE)
PDF_PATTERN_RE = re.compile(r'\s[0-9]*\s[km]b\s*pdf', re.IGNORECASE)
PUNCT_RE = re.compile(r'[^\w\s-]')  # match punctuation
MULTISPACE_RE = re.compile(r'\s+')

stopword_re_str = r""
for word in STOPWORDS:
	stopword_re_str += r'\b' + word + r'\b|'
stopword_re = re.compile(stopword_re_str[:-1]) # The negative 1 is for the fencepost |

NON_FINANCIAL_ORG_TERMS = [
    'university', 'college', 'school', 'institute', 'academy', 
    'hospital', 'medical center', 'health system', 'center', 
    'commission', 'authority', 'association', 'society', 
    'foundation', 'transportation services', 'district', 
    'chamber', 'commerce', 'library', 'museum', 'public', 
    'city', 'county', 'town', 'government', 'state', 'federal',
    'ministry', 'department', 'office'
]

NON_FINANCIAL_RE = re.compile(r'\b(' + '|'.join(NON_FINANCIAL_ORG_TERMS) + r')\b', re.IGNORECASE)

BASE_DIR = "/Users/aawesomez/Documents/UROP/NLP-regextable/"
# BASE_DIR = "/Users/jameschen/Team Name Dropbox/James Chen/JLW-FINREG-PARTICIPATION/"
# BASE_DIR = "/Users/jameschen/Documents/Code/JLW-FINREG-PARTICIPATION/"
# DB_PATH = BASE_DIR + "data/master.sqlite"
DB_PATH = BASE_DIR + "Data/master.sqlite"
# LAST_SAVE_DATASET_DATE = "20210824"
LAST_SAVE_DATASET_DATE = "20220402" # Needs to be set to the last date the 'rebuild datasets' part of this code was run

# Function to calculate longest common substring, from https://www.geeksforgeeks.org/print-longest-common-substring/
# function to find and print 
# the longest common substring of
# X[0..m-1] and Y[0..n-1]
def get_longest_common_substring(X, Y, m, n):
 
    # Create a table to store lengths of
    # longest common suffixes of substrings.
    # Note that LCSuff[i][j] contains length
    # of longest common suffix of X[0..i-1] and
    # Y[0..j-1]. The first row and first
    # column entries have no logical meaning,
    # they are used only for simplicity of program
    LCSuff = [[0 for i in range(n + 1)]
                 for j in range(m + 1)]
 
    # To store length of the
    # longest common substring
    length = 0
 
    # To store the index of the cell
    # which contains the maximum value.
    # This cell's index helps in building
    # up the longest common substring
    # from right to left.
    row, col = 0, 0
 
    # Following steps build LCSuff[m+1][n+1]
    # in bottom up fashion.
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 or j == 0:
                LCSuff[i][j] = 0
            elif X[i - 1] == Y[j - 1]:
                LCSuff[i][j] = LCSuff[i - 1][j - 1] + 1
                if length < LCSuff[i][j]:
                    length = LCSuff[i][j]
                    row = i
                    col = j
            else:
                LCSuff[i][j] = 0
 
    # if true, then no common substring exists
    if length == 0:
        return ""
 
    # allocate space for the longest
    # common substring
    resultStr = ['0'] * length
 
    # traverse up diagonally form the
    # (row, col) cell until LCSuff[row][col] != 0
    while LCSuff[row][col] != 0:
        length -= 1
        resultStr[length] = X[row - 1] # or Y[col-1]
 
        # move diagonally up to previous cell
        row -= 1
        col -= 1
 
    # required longest common substring
    longest_common_substring = ''.join(resultStr)

    return longest_common_substring


# Function from Brad Hackinen's NAMA
def basicHash(s):
    '''
    A simple case and puctuation-insensitive hash
    '''
    s = s.lower()
    s = re.sub(and_re,' and ',s)
    s = re.sub(punc1_re,'',s)
    s = re.sub(punc2_re,' ',s)
    s = s.strip()

    return s

# Function from Brad Hackinen's NAMA
def corpHash(s):
    '''
    A hash function for corporate subsidiaries
    Insensitive to
        -case & punctation
        -'the' prefix
        -common corporation suffixes, including 'holding co'
    '''
    s = basicHash(s)
    if s.startswith('the '):
        s = s[4:]

    s = re.sub(corp_re,'',s,count=1)

    return s

# function to clean org names
def clean_fin_org_names(name: str) -> str:
    if name is None or not isinstance(name, str) or name == "NA":
        return ""
    
    # James strip metadata from name
    name = name.split(',')[0]
    #Remove patterns like "10 kb pdf"
    name = PDF_PATTERN_RE.sub("", name)

    #Unicode and punctuation cleanup
    name = corp_simplify_utils.normalize_unicode(name)
    name = PUNCT_RE.sub(" ", name)

    #Remove corporate suffixes and stopwords and non-financial entity
    name = CORP_SUFFIX_RE.sub("", name)
    name = NON_FINANCIAL_RE.sub("", name)
    name = STOPWORD_RE.sub("", name)

    #Normalize spacing and lowercase
    name = MULTISPACE_RE.sub(" ", name).strip().lower()

    return name

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/stevenkang/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
# Locate data directory and read in data files
import os
from pathlib import Path

current_dir = Path(os.getcwd()).parent
data_dir = current_dir / 'data'
data_dir = data_dir.resolve()

try:
    compustat_df = pd.read_csv(data_dir / 'CompustatNames.csv')
    cik_df = pd.read_csv(data_dir / 'CIK.csv')
    fdic_df = pd.read_csv(data_dir / 'FDIC_clean.csv') # Using your 'FDIC_clean.csv'
    sec_df = pd.read_csv(data_dir / 'SEC_Institutions.csv')
except FileNotFoundError as e:
    print(f"Error loading file: {e}")
    print(f"Please make sure your CSV files are in the directory: {data_dir}")
    exit()
    

In [18]:
# cleaning and standardizing organization names
compustat_df['std_name'] = compustat_df['conm'].apply(clean_fin_org_names)
fdic_df['std_name'] = fdic_df['NAME'].apply(clean_fin_org_names)
sec_df['std_name'] = sec_df['Name'].apply(clean_fin_org_names)
cik_df['std_name'] = cik_df['company_name'].apply(clean_fin_org_names)

In [19]:
print(cik_df.shape)
cik_df.head(10)

(870051, 4)


,Unnamed: 0,company_name,cik,std_name
0,0,!J INC,1438823.0,j
1,1,"#1 A LIFESAFER HOLDINGS, INC.",1509607.0,1 a lifesafer holdings
2,2,#1 ARIZONA DISCOUNT PROPERTIES LLC,1457512.0,1 arizona discount properties
3,3,#1 PAINTBALL CORP,1433777.0,1 paintball
4,4,$ LLC,1427189.0,
5,5,"$AVY, INC.",1655250.0,avy
6,6,& S MEDIA GROUP LLC,1447162.0,s media group
7,7,&TV COMMUNICATIONS INC.,1479357.0,tv communications
8,8,"&VEST DOMESTIC FUND II KPIV, L.P.",1802417.0,vest domestic fund ii kpiv
9,9,&VEST DOMESTIC FUND II LP,1800903.0,vest domestic fund ii lp


In [15]:
print(sec_df.shape)
sec_df.head(10)

(13737, 9)


,index,CIK,Ticker,Name,Exchange,SIC,Business,Incorporated,IRS
0,0,1090872,A,Agilent Technologies Inc,NYSE,3825.0,CA,DE,770518772.0
1,1,4281,AA,Alcoa Inc,NYSE,3350.0,PA,PA,250317820.0
2,2,1332552,AAACU,Asia Automotive Acquisition Corp,NaN,6770.0,DE,DE,203022522.0
3,3,1287145,AABB,Asia Broadband Inc,OTC,8200.0,GA,NV,721569126.0
4,4,1024015,AABC,Access Anytime Bancorp Inc,NaN,6035.0,NM,DE,850444597.0
5,5,1099290,AAC,Sinocoking Coal & Coke Chemical Industries Inc,NASDAQ,3312.0,F4,FL,593404233.0
6,6,1264707,AACC,Asset Acceptance Capital Corp,NaN,6153.0,MI,NaN,800076779.0
7,7,849116,AACE,Ace Cash Express Inc,NaN,6099.0,TX,TX,752142963.0
8,8,1409430,AAGC,All American Gold Corp,OTC,1040.0,IN,WY,260665571.0
9,9,948846,AAI,Airtran Holdings Inc,NaN,4512.0,FL,NV,582189551.0


In [20]:
print(compustat_df.shape)
compustat_df.head(100)

(19581, 13)


,Unnamed: 0,gvkey,conm,tic,cusip,cik,sic,naics,gsubind,gind,year1,year2,std_name
0,1,1004,AAR CORP,AIR,000361105,1750.0,5080.0,423860.0,20101010.0,201010.0,1965,2020,aar
1,2,1013,ADC TELECOMMUNICATIONS INC,ADCT.1,000886309,61478.0,3661.0,334210.0,45201020.0,452010.0,1974,2010,adc telecommunications
2,3,1045,AMERICAN AIRLINES GROUP INC,AAL,02376R102,6201.0,4512.0,481111.0,20302010.0,203020.0,1950,2021,american airlines group
3,4,1050,CECO ENVIRONMENTAL CORP,CECE,125141101,3197.0,3564.0,333413.0,20201050.0,202010.0,1974,2021,ceco environmental
4,5,1062,ASA GOLD AND PRECIOUS METALS,ASA,G3156P103,1230869.0,6799.0,523999.0,40203010.0,402030.0,1966,2021,asa gold precious metals
...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,1690,APPLE INC,AAPL,037833100,320193.0,3663.0,334220.0,45202030.0,452020.0,1980,2021,apple
96,97,1704,APPLIED MATERIALS INC,AMAT,038222105,6951.0,3559.0,333242.0,45301010.0,453010.0,1972,2021,applied materials
97,98,1706,ENERPAC TOOL GROUP CORP,EPAC,292765104,6955.0,3533.0,333132.0,20106020.0,201060.0,1974,2021,enerpac tool group
98,99,1712,TRECORA RESOURCES,TREC,894648104,7039.0,2911.0,324110.0,15101010.0,151010.0,1976,2021,trecora resources


In [17]:
print(fdic_df.shape)
fdic_df.head(10)

(25670, 15)


,NAME,NAMEHCR,STALP,STNAME,BKCLASS,ASSET,CERT,FED_RSSD,org_name,commented,Commented,mean_ASSET,median_ASSET,mean_ASSET_type,median_ASSET_type
0,The Southington Bank and Trust Company,NaN,CT,Connecticut,NM,4.857000e+07,4,573401,the southington bank and trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000
1,Colonial Bank of Waterbury,NaN,CT,Connecticut,NM,6.246550e+08,6,148304,colonial bank of waterbury,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000
2,Fleet Bank of Maine,NaN,ME,Maine,SM,1.699404e+09,8,422406,fleet bank of maine,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000
3,Union Trust Company,NaN,ME,Maine,SM,5.391690e+08,9,563907,union trust company,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000
4,Northeast Bank of Sanford,NaN,ME,Maine,SM,5.569200e+07,10,112109,northeast bank of sanford,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000
5,State Street Bank and Trust Company,State Street Corporation,MA,Massachusetts,SM,2.335429e+11,14,35301,state street bank and trust company,True,Commented,2.565979e+09,145717500,3.941421e+09,218865500
6,BayBank Harvard Trust Company,NaN,MA,Massachusetts,NM,1.435310e+09,18,852209,baybank harvard trust company,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000
7,Durfee Attleboro Bank,NaN,MA,Massachusetts,NM,3.468080e+08,20,202402,durfee attleboro bank,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000
8,Bank of New England - North Shore,NaN,MA,Massachusetts,SM,1.532740e+08,21,658205,bank of new england - north shore,False,Did not comment,9.155463e+08,95742500,9.804661e+08,131926000
9,Shawmut Bank of Franklin County,NaN,MA,Massachusetts,NM,1.597270e+08,22,661205,shawmut bank of franklin county,False,Did not comment,9.155463e+08,95742500,3.082197e+08,76186000


In [21]:
# Test if CIK is already in SEC
sec_ciks = set(sec_df['CIK'].dropna())
cik_ciks = set(cik_df['cik'].dropna())

print(f"CIKs in sec_df: {len(sec_ciks)}")
print(f"CIKs in cik_df: {len(cik_ciks)}")
print(f"Is SEC_Institutions.csv a subset of CIK.csv? {sec_ciks.issubset(cik_ciks)}")

compustat_ciks = set(compustat_df['cik'].dropna())
print(f"CIKs in compustat_df: {len(compustat_ciks)}")
print(f"Is CompustatNames.csv a subset of CIK.csv? {compustat_ciks.issubset(cik_ciks)}")


CIKs in sec_df: 13737
CIKs in cik_df: 806225
Is SEC_Institutions.csv a subset of CIK.csv? True
CIKs in compustat_df: 12835
Is CompustatNames.csv a subset of CIK.csv? False


SEC_Institutions.csv is a subset of CIK.csv --- > Don't need to merge SEC into the crosswalk. 

In [25]:
# renaming columns for consistency
compustat_temp = compustat_df[['std_name', 'conm', 'cik']].rename(columns={'conm': 'raw_name'})
compustat_temp['source'] = 'compustat'

fdic_temp = fdic_df[['std_name', 'NAME']].rename(columns={'NAME': 'raw_name'})
fdic_temp['source'] = 'fdic'

cik_temp = cik_df[['std_name', 'company_name', 'cik']].rename(columns={'company_name': 'raw_name'})
cik_temp['source'] = 'cik'

# Combine all into one long dataframe
all_names_df = pd.concat([compustat_temp, fdic_temp, cik_temp], ignore_index=True)

# Drop any rows where cleaning failed (no std_name)
all_names_df = all_names_df.dropna(subset=['std_name'])
# Remove any empty std_name entries
all_names_df = all_names_df[all_names_df['std_name'] != ""]

# Cleaning up CIKs and FED_RSSD to be strings without decimal points
for col in ['CIK', 'FED_RSSD']:
    if col in all_names_df.columns:
        # Convert to string after converting to int to remove any decimal points
        all_names_df[col] = all_names_df[col].dropna().astype(float).astype(int).astype(str)

print(f"Total entries to match: {len(all_names_df)}")
all_names_df.head(20)

Total entries to match: 915289


,std_name,raw_name,cik,source
0,aar,AAR CORP,1750.0,compustat
1,adc telecommunications,ADC TELECOMMUNICATIONS INC,61478.0,compustat
2,american airlines group,AMERICAN AIRLINES GROUP INC,6201.0,compustat
3,ceco environmental,CECO ENVIRONMENTAL CORP,3197.0,compustat
4,asa gold precious metals,ASA GOLD AND PRECIOUS METALS,1230869.0,compustat
5,avx,AVX CORP,859163.0,compustat
6,pinnacle west capital,PINNACLE WEST CAPITAL CORP,764622.0,compustat
7,prog holdings,PROG HOLDINGS INC,1808834.0,compustat
8,abbott laboratories,ABBOTT LABORATORIES,1800.0,compustat
9,servidyne,SERVIDYNE INC,1923.0,compustat


In [26]:
grouped_by_cik_id = all_names_df.groupby('cik')
confident_matches = []

for cik_value, group in grouped_by_cik_id:
    if len(group) > 1:
        # Aggregate the data based on cik
        keys = {
            'cik': group['cik'].dropna().unique().tolist(),
            # Now aggregate the std_name to see all variations found for cik
            'standardized_names': '|'.join(group['std_name'].dropna().unique()),
            # Aggregate other fields as before
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique())
        }
        confident_matches.append(keys)
        
pd.set_option('display.max_colwidth', None)
confident_crosswalk_id_based = pd.DataFrame(confident_matches)
print(f"Found {len(confident_crosswalk_id_based)} confident entity clusters.")

Found 56760 confident entity clusters.


In [27]:
confident_crosswalk_id_based.head(10)

,cik,standardized_names,aliases,sources
0,[1750.0],aar,AAR CORP,"compustat,cik"
1,[1800.0],abbott laboratories,ABBOTT LABORATORIES,"compustat,cik"
2,[1841.0],abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik
4,[1860.0],thomson richard william bd|thomson,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik
5,[1904.0],abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik
6,[1918.0],abrams|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik
7,[1923.0],servidyne|abrams industries,"SERVIDYNE INC|ABRAMS INDUSTRIES INC|SERVIDYNE, INC.","compustat,cik"
8,[1961.0],worlds|academic computer systems|worlds com,"WORLDS INC|ACADEMIC COMPUTER SYSTEMS INC|WORLDS COM INC|WORLDS.COM, INC.","compustat,cik"
9,[2034.0],aceto,ACETO CORP,"compustat,cik"


Able to create a table of 56760 companies based on CIK id.

In [28]:
# Find all the rows where there is duplicate CIK in one of the dataframes
# In other words, there are multiple aliases, but coming from the same source

confident_crosswalk_id_based[
    (confident_crosswalk_id_based['aliases'].str.contains('\|')) & 
    (~confident_crosswalk_id_based['sources'].str.contains(r','))
].head(20)

,cik,standardized_names,aliases,sources
2,[1841.0],abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik
4,[1860.0],thomson richard william bd|thomson,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik
5,[1904.0],abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik
6,[1918.0],abrams|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik
11,[2093.0],acme metals de|acme metals,ACME METALS INC /DE/|ACME METALS INC/,cik
13,[2110.0],acorn investment trust|columbia acorn trust|liberty acorn trust,ACORN INVESTMENT TRUST|COLUMBIA ACORN TRUST|LIBERTY ACORN TRUST,cik
17,[2310.0],am international|multigraphics,AM INTERNATIONAL INC|MULTIGRAPHICS INC,cik
18,[2380.0],administrative data management ta|foresters investor services ta,ADMINISTRATIVE DATA MANAGEMENT CORP /TA|ADMINISTRATIVE DATA MANAGEMENT CORP /TA|FORESTERS INVESTOR SERVICES INC/TA,cik
21,[2554.0],aei securities bd|aei securities,"AEI SECURITIES INC /BD|AEI SECURITIES, INC.",cik


There appears to be many aliases for the same CIK ID in the cik.csv file

In [29]:
# need to isolate remaining data that couldn't be matched by CIK
# checks the size of each group by CIK
cik_group_sizes = all_names_df.groupby('cik')['cik'].transform('size')
processed_rows_mask = (cik_group_sizes > 1)
remaining_df = all_names_df[~processed_rows_mask].copy()

total_rows = len(all_names_df)
processed_rows_count = processed_rows_mask.sum()
remaining_rows_count = len(remaining_df)
print(f"Total rows: {total_rows}")
print(f"Processed rows (matched by CIK): {processed_rows_count}")
print(f"Remaining rows to process: {remaining_rows_count}")

Total rows: 915289
Processed rows (matched by CIK): 133395
Remaining rows to process: 781894


In [30]:
# Now group by std_name to find additional matches
grouped_by_std_name = remaining_df.groupby('std_name')
confident_matches_std_name = []

for name, group in grouped_by_std_name:
    # A "match" means this std_name appeared in more than one row
    if len(group) > 1:
        # Aggregate all unique keys and aliases
        keys = {
            'std_name': name,
            'aliases': '|'.join(group['raw_name'].dropna().unique()),
            'sources': ','.join(group['source'].unique()),
            'CIK': group['cik'].dropna().unique().tolist(),
        }
        confident_matches_std_name.append(keys)
pd.set_option('display.max_rows', None)
confident_crosswalk_std_name = pd.DataFrame(confident_matches_std_name)
print(f"  Found {len(confident_crosswalk_std_name)} additional clusters based on std_name.")
confident_crosswalk_std_name.head(10)

  Found 14708 additional clusters based on std_name.


,std_name,aliases,sources,CIK
0,-depth partners global equities,IN-DEPTH PARTNERS GLOBAL EQUITIES LLC|IN-DEPTH PARTNERS GLOBAL EQUITIES LTD.,cik,"[1882352.0, 1882346.0]"
1,-investment income fund,"CO-INVESTMENT INCOME FUND, L.P. - NON-US SERIES|CO-INVESTMENT INCOME FUND, L.P. - US TAX EXEMPT SERIES|CO-INVESTMENT INCOME FUND, L.P. - US TAXABLE SERIES",cik,"[1671812.0, 1689685.0, 1671832.0]"
2,-operative bank concord,The Co-operative Bank of Concord,fdic,[]
3,0x fund i,"0X FUND I, A SERIES OF DEEP VENTURES SYNDICATE, LP|0X FUND I, A SERIES OF KINGS COUNTY VENTURES, LP",cik,"[1912493.0, 1843213.0]"
4,1 main capital partners,"1 MAIN CAPITAL PARTNERS, LP|1 MAIN CAPITAL PARTNERS, LTD.",cik,"[1728406.0, 1880919.0]"
5,10 fund i,"10 FUND I, A SERIES OF EPAKON CAPITAL, LP|10 FUND I, A SERIES OF HACK VC, LP",cik,"[1917519.0, 1860371.0]"
6,10x capital gaingels diversity fund,"10X CAPITAL / GAINGELS DIVERSITY FUND, LP - A3|10X CAPITAL / GAINGELS DIVERSITY FUND, LP - A4",cik,"[1900341.0, 1935728.0]"
7,15five,"15FIVE, A SERIES OF JASON'S SYNDICATE, LLC|15FIVE, INC.",cik,"[1756516.0, 1756071.0]"
8,180 jamaica,180 JAMAICA CORP.|180 JAMAICA INC,cik,"[1294080.0, 1051663.0]"
9,180s,180S INC|180S LLC,cik,"[1305605.0, 1295552.0]"


In [31]:
# Utility functions to combine pipe-separated and comma-separated strings
# Coutesy of Google Gemini

def combine_pipes(a, b):
    """Combines two pipe-separated strings with unique, sorted values."""
    a_set = set(str(a).split('|')) if pd.notna(a) and str(a).strip() != '' else set()
    b_set = set(str(b).split('|')) if pd.notna(b) and str(b).strip() != '' else set()
    return '|'.join(sorted(a_set.union(b_set)))

def combine_commas(a, b):
    """Combines two comma-separated strings with unique, sorted values."""
    a_set = set(str(a).split(',')) if pd.notna(a) and str(a).strip() != '' else set()
    b_set = set(str(b).split(',')) if pd.notna(b) and str(b).strip() != '' else set()
    return ','.join(sorted(a_set.union(b_set)))

def combine_lists(list_a, list_b):
    """Combines two lists with unique, sorted values."""
    set_a = set(list_a) if isinstance(list_a, list) else set()
    set_b = set(list_b) if isinstance(list_b, list) else set()
    return sorted(list(set_a.union(set_b)))

In [32]:
# Copy the master crosswalk and new clusters
master_crosswalk = confident_crosswalk_id_based.copy() 
new_clusters = confident_crosswalk_std_name.copy()

# Add the 'match_type' column to the master table
# This will help us track how each cluster was formed
master_crosswalk['match_type'] = 'cik_cluster'

# Create a lookup DataFrame from the master table
# explode the standardized_names into single names
# 'A|B|C' ---> into three rows: A, B, and C
# We also reset the index to get a 'master_index' column we can use to map back
master_lookup = master_crosswalk.reset_index().rename(columns={'index': 'master_index'})
# splits into a list of names
master_lookup['std_name_single'] = master_lookup['standardized_names'].str.split('|')
master_lookup = master_lookup.explode('std_name_single')
master_lookup = master_lookup[['std_name_single', 'master_index']].drop_duplicates()


# Keep track of which new_cluster rows we've successfully merged
new_clusters['processed'] = False

print(f"Starting merge. Enriching {len(master_crosswalk)} CIK clusters with {len(new_clusters)} std_name clusters...")

for new_idx, new_row in new_clusters.iterrows():
    
    std_name_to_match = new_row['std_name']
    
    # Find all master_crosswalk rows that already contain this std_name
    matching_master_indices = master_lookup[
        master_lookup['std_name_single'] == std_name_to_match
    ]['master_index'].unique()

    if len(matching_master_indices) > 0:
        # Found a match, so get the first matching row's index in the master_crosswalk
        master_idx = matching_master_indices[0] 
        
        # Combine Aliases
        master_crosswalk.loc[master_idx, 'aliases'] = combine_pipes(
            master_crosswalk.loc[master_idx, 'aliases'], new_row['aliases']
        )

        # Combine Sources
        master_crosswalk.loc[master_idx, 'sources'] = combine_commas(
            master_crosswalk.loc[master_idx, 'sources'], new_row['sources']
        )

        # Combine CIKs - Use .at to assign a list to a single cell
        master_crosswalk.at[master_idx, 'cik'] = combine_lists(
            master_crosswalk.loc[master_idx, 'cik'], new_row['CIK']
        )

        # Combine standardized_names
        master_crosswalk.loc[master_idx, 'standardized_names'] = combine_pipes(
            master_crosswalk.loc[master_idx, 'standardized_names'], new_row['std_name']
        )
        
        # Update match_type
        master_crosswalk.loc[master_idx, 'match_type'] = combine_commas(
            master_crosswalk.loc[master_idx, 'match_type'], 'std_name_cluster'
        )
        # Mark this new_cluster row as processed
        new_clusters.loc[new_idx, 'processed'] = True
        
        # Note: If a std_name links two *different* CIK clusters, this code
        # merges it into the *first* one it finds. 

# Append Unmatched Clusters
# Find all rows from new_clusters that were *not* processed (i.e., no match)
unmatched_clusters = new_clusters[new_clusters['processed'] == False].copy()

print(f"Enriched {new_clusters['processed'].sum()} existing clusters.")
print(f"Found {len(unmatched_clusters)} brand new clusters to append.")

# Rename columns to match the master_crosswalk before appending
unmatched_clusters = unmatched_clusters.rename(columns={
    'std_name': 'standardized_names',
    'CIK': 'cik'
})
# Add the new match_type, 'std_name_cluster' means that this cluster was formed based on std_name matching
unmatched_clusters['match_type'] = 'std_name_cluster'

master_crosswalk = master_crosswalk.drop(columns=['processed'], errors='ignore')
unmatched_clusters = unmatched_clusters.drop(columns=['processed'], errors='ignore')

final_enriched_crosswalk = pd.concat(
    [master_crosswalk, unmatched_clusters], 
    ignore_index=True
)

print(f"\nFinal crosswalk has {len(final_enriched_crosswalk)} total clusters.")

final_enriched_crosswalk.head(20)

Starting merge. Enriching 56760 CIK clusters with 14708 std_name clusters...
Enriched 759 existing clusters.
Found 13949 brand new clusters to append.

Final crosswalk has 70709 total clusters.


,cik,standardized_names,aliases,sources,match_type
0,[1750.0],aar,AAR CORP,"compustat,cik",cik_cluster
1,[1800.0],abbott laboratories,ABBOTT LABORATORIES,"compustat,cik",cik_cluster
2,[1841.0],abel noser bd|abel noser,ABEL NOSER CORP /BD|ABEL/NOSER CORP.,cik,cik_cluster
3,[1853.0],aberdeen idaho mining|motivnation,"ABERDEEN IDAHO MINING CO|MOTIVNATION, INC.",cik,cik_cluster
4,[1860.0],thomson richard william bd|thomson,"THOMSON RICHARD WILLIAM /BD|THOMSON, RICHARD WILLIAM",cik,cik_cluster
5,[1904.0],abraham bd|abraham|abraham securities corporation,"ABRAHAM & CO INC /BD|ABRAHAM & CO., INC.|ABRAHAM SECURITIES CORPORATION",cik,cik_cluster
6,[1918.0],abrams|homeland securities financial services group|merchanthouse securities|wizer financial,"ABRAMS, ALLAN EDWARD|HOMELAND SECURITIES FINANCIAL SERVICES GROUP, INC.|THE MERCHANTHOUSE SECURITIES, INC.|WIZER FINANCIAL. INC.",cik,cik_cluster
7,[1923.0],servidyne|abrams industries,"SERVIDYNE INC|ABRAMS INDUSTRIES INC|SERVIDYNE, INC.","compustat,cik",cik_cluster
8,[1961.0],worlds|academic computer systems|worlds com,"WORLDS INC|ACADEMIC COMPUTER SYSTEMS INC|WORLDS COM INC|WORLDS.COM, INC.","compustat,cik",cik_cluster
9,[2034.0],aceto,ACETO CORP,"compustat,cik",cik_cluster


In [34]:
print(final_enriched_crosswalk.head(3).to_markdown(index=False))

| cik      | standardized_names       | aliases                                                                      | sources       | match_type   |
|:---------|:-------------------------|:-----------------------------------------------------------------------------|:--------------|:-------------|
| [1750.0] | aar                      | AAR CORP                                                                     | compustat,cik | cik_cluster  |
| [1800.0] | abbott laboratories      | ABBOTT LABORATORIES                                                          | compustat,cik | cik_cluster  |
| [1841.0] | abel noser bd|abel noser | ABEL NOSER CORP                                         /BD|ABEL/NOSER CORP. | cik           | cik_cluster  |


In [ ]:
final_enriched_crosswalk[final_enriched_crosswalk['match_type'].str.contains('cik_cluster,std_name_cluster')]

In [ ]:
# Create an ID column, which is just the row index plus one
# Create the 7-digit numeric part, padded with leading zeros
numeric_id = final_enriched_crosswalk.index.astype(str).str.zfill(7)
final_enriched_crosswalk['entity_id'] = 'EN' + numeric_id

print(f"Added new 'entity_id' column with 7 digits.")
final_enriched_crosswalk.head()

In [ ]:
"""
Process to get data that has not been matched by std_name or CIK
- get a list of all the std_names that have been matched
- filter the all_names_df to get rows that have not been matched by CIK or std_name
- this will give us the remaining unmatched rows to process further
"""

processed_names_series = final_enriched_crosswalk['standardized_names'].dropna().str.split('|')
processed_std_names_set = set(processed_names_series.explode().unique())

print(f"Found {len(processed_std_names_set)} unique standardized names already in the crosswalk.")

# True if the name is NOT in the set, False if it is
unmatched_mask = ~all_names_df['std_name'].isin(processed_std_names_set)
unmatched_for_fuzzy = all_names_df[unmatched_mask].copy()
unmatched_for_fuzzy = unmatched_for_fuzzy.reset_index(drop = True)

print(f"Original data: {len(all_names_df)} rows")
print(f"Data remaining for fuzzy matching: {len(unmatched_for_fuzzy)} rows")

unmatched_for_fuzzy.head(100)


In [ ]:
# Create a testing mode to sample data for faster fuzzy matching
TESTING_MODE = True
SAMPLE_FRAC = 0.05 # choose percent of data to sample in testing mode

if TESTING_MODE:
    print(f"--- RUNNING FUZZY MATCHING IN TESTING MODE (Sample: {SAMPLE_FRAC*100}%) ---")
    
    # Take a random sample of your 'unmatched_for_fuzzy' DataFrame
    unmatched_for_fuzzy = unmatched_for_fuzzy.sample(frac=SAMPLE_FRAC, random_state=42)
    
    print(f"  'unmatched_for_fuzzy' sampled to: {len(unmatched_for_fuzzy)} rows")
    print("-------------------------------------------------")
    
else:
    print(f"--- RUNNING FUZZY MATCHING IN FULL PRODUCTION MODE ---")
    print(f"  'unmatched_for_fuzzy' full size: {len(unmatched_for_fuzzy)} rows")
    print("-------------------------------------------------")

In [ ]:
# Imported function from the regextable-python repository
# get_match_candidate_score get the matching score between two names
def get_match_candidate_score(frequency_dict, org_name, candidate_match_name):
    if not isinstance(org_name, str):
        org_name = ""
    if not isinstance(candidate_match_name, str):
        candidate_match_name = ""
    
    if not org_name or not candidate_match_name:
        return 0.0
    
    org_tokens = org_name.split(' ')
    
    # tokenize the candidate match
    candidate_match_tokens = set(candidate_match_name.split(" "))

    max_dist = 1

    # Calculate the match score
    total_inverse_frequency = 0
    total_matching_inverse_frequency = 0
    tokenized_name = org_tokens
    for token in tokenized_name:
        token_frequency = frequency_dict.get(token, 999999) # if token not found, give high frequency to ignore it
        token_inverse_frequency = 1.0/token_frequency
        total_inverse_frequency += token_inverse_frequency

        best_token_similarity = 0.0
        
        for candidate_token in candidate_match_tokens:
            if not token or not candidate_token:
                continue

            dist = damerau_levenshtein_distance(token, candidate_token)

            if dist <= max_dist:
                max_len = max(len(token), len(candidate_token))
                if max_len == 0: continue
                similarity = 1.0 - (dist / max_len)

                best_token_similarity = max(best_token_similarity, similarity)
        if best_token_similarity > 0.0:
            total_matching_inverse_frequency += token_inverse_frequency * best_token_similarity
    
    try:
        match_score = total_matching_inverse_frequency / total_inverse_frequency
    except ZeroDivisionError:
        match_score = 0.0
    
    #Multiplicative DL Penalty
    m = len(org_name)
    n = len(candidate_match_name)
    if m == 0 or n == 0: return 0.0

    dl_distance = damerau_levenshtein_distance(org_name, candidate_match_name)
    normalized_dl = dl_distance / max(m, n)

    final_score = match_score * (1 - normalized_dl)
    return max(0.0, final_score)
        
        
def clean_match_score(x):
    if x is None or x is np.nan or pd.isnull(x) or x == "":
        return np.nan
    elif isinstance(x, str) and not x.isnumeric():
        unit_multiplier = 1
        if "B" in x:
            x = x[:-1]
            unit_multiplier = 1000000000
        if "M" in x:
            x = x[:-1]
            unit_multiplier = 1000000
        if "K" in x:
            x = x[:-1]
            unit_multiplier = 1000
        x = x.replace(",", "")
        try:
            x = float(x) * unit_multiplier
            return x
        except:
            return np.nan
    else:
        return float(x)


In [ ]:
from collections import defaultdict

frequency_df = pd.DataFrame()
frequency_df['std_name'] = unmatched_for_fuzzy['std_name']
frequency_dict = defaultdict(int)

# Iterate over the clean organization names
for name in frequency_df['std_name']:
    # Split the name into tokens (words)
    tokens = name.split(' ')

    # Update the count for each token
    for token in tokens:
        # Filter out empty strings that might result from extra spaces
        if token:
            frequency_dict[token] += 1

# Convert defaultdict back to a standard dict for the function
frequency_data = dict(frequency_dict)

print("Printing the frequency_dict")
print(frequency_data)

In [ ]:
unmatched_for_fuzzy = unmatched_for_fuzzy.reset_index(drop = True)
unmatched_for_fuzzy = unmatched_for_fuzzy.drop(columns = ['tic', 'cusip'])
unmatched_for_fuzzy.head(10)

In [ ]:
# import unittest
# from pandas.testing import assert_frame_equal

# # index_of_entity is the index of the row you want to add from df_original to df_new
# # df_new needs to have a column 'original_index 'that keeps track of the index of where a std_name is from in df_original 
# #TODO: change this logic so that the column of orginal index is a dictionary of the indexes. 

# def add_entity_to_df(df_new: pd.DataFrame, df_original: pd.DataFrame, index_of_entity: int): 
#     # Create the mask to find existing entries
#     mask = df_new['original_index'].apply(lambda x: index_of_entity in x)
#     new_std_name = df_original.iloc[index_of_entity]['std_name']

#     # Case 1: there is already the entity in the new data frame
#     if mask.any():
#         current_std_name = df_new.loc[mask, 'std_name']
#         new_combined_std_name = current_std_name + ', ' + new_std_name
#         df_new.loc[mask, 'std_name'] = new_combined_std_name 
#         df_new.loc[mask, 'original_index'] = df_new.loc[mask, 'original_index'].apply(lambda x: x + [index_of_entity])
        
#     # Case 2: add the entitiy as a new row in the data frame  
#     else: 
#         new_row_series = df_original.iloc[index_of_entity].copy()
#         new_row_series['original_index'] = [index_of_entity] 
#         df_new = pd.concat([df_new, new_row_series.to_frame().T], ignore_index=True) 
      
#     return df_new

In [ ]:
from union_find import UnionFind

# initialize union-find here
uf = UnionFind(len(unmatched_for_fuzzy)) 
print(len(uf.parent))

# uf.unite(1, 1000)
# uf.unite(1, 30)
# set_i = uf.find(1000)
# set_j = uf.find(1)
# set_k = uf.find(30)

# print(f'set_i: {set_i}, set_j: {set_j}, set_k: {set_k}')
# print(uf.find(200))

In [ ]:
from rapidfuzz import process, fuzz
from rapidfuzz.distance import JaroWinkler
# score = fuzz.token_set_ratio("bank of american", 'bank of america')
# score

# score = JaroWinkler.similarity("bank of american", "bank of amercia")
# score

In [ ]:
# count = 0
for i in range(len(unmatched_for_fuzzy)):
   # count += 1 
   # if count % 10 == 0:
   #    print(count)
      
   item_i = unmatched_for_fuzzy.iloc[i]
   for j in range(i+1, len(unmatched_for_fuzzy)):
      item_j = unmatched_for_fuzzy.iloc[j]
      
      # add union-find here
      set_i = uf.find(i)
      set_j = uf.find(j)
      # When two index are in the same set, they already matched, skip
      if set_i == set_j:
         continue 
      
      # score = get_match_candidate_score(frequency_dict, item_i['std_name'], item_j['std_name']) * 100
      # score = fuzz.token_set_ratio(item_i['std_name'], item_j['std_name'])
      score = JaroWinkler.similarity(item_i['std_name'], item_j['std_name']) * 100
      # if j % 3 == 0: 
      #    print(f"i is {i} and j is {j}, score is {score}")
      
      if score < 90: 
         # print(f"Warn — itemj name: {item_j['std_name']}, itemi name: {item_i['std_name']}")
         continue
      
      uf.unite(i, j)
      # print(f"Match: i is {i} and j is {j}")
      # print(f"item_i: {item_i['std_name']}, item_j: {item_j['std_name']}")
      
      
   if i== 20:
      break

In [ ]:
# The dictionary of parents and their row lists so that indices in the same list will be turned into a series
# that will be merged into the new matched_df
index_dict = defaultdict(list)
for i in range(len(uf.parent)):
    current_parent = uf.find(i)
    index_dict[current_parent].append(i)
index_dict = dict(index_dict)
print(index_dict)
print(len(index_dict))

In [ ]:
# TODO: Implement the for-loop that goes through all the keys in the sets and appends a new series to matched_df
cols = ['standardized_names', 'aliases', 'cik', 'sources', 'original_index', 'score']
matched_df = pd.DataFrame(columns = cols)

count = 0
for key in index_dict:
    count += 1 
    if count % 100 == 0:
        print(count)
    row_list = index_dict[key]
    new_std_name = ""
    new_aliases = ""
    new_sources = ""
    new_cik = []
    new_original_indices = []
    
    for i in range(len(row_list)):
        row = row_list[i]
        if i == len(row_list) - 1:
            new_std_name += unmatched_for_fuzzy.iloc[row]['std_name']
            new_aliases += unmatched_for_fuzzy.iloc[row]['raw_name']
            new_sources += unmatched_for_fuzzy.iloc[row]['source']
            new_cik.append(unmatched_for_fuzzy.iloc[row]['cik'])
            new_original_indices.append(row)
        else:
            new_std_name += unmatched_for_fuzzy.iloc[row]['std_name'] + "|"
            new_aliases += unmatched_for_fuzzy.iloc[row]['raw_name'] + "|"
            new_sources += unmatched_for_fuzzy.iloc[row]['source'] + ","
            new_cik.append(unmatched_for_fuzzy.iloc[row]['cik']) 
            new_original_indices.append(row)  
        
        combined_row_dict = {"standardized_names": new_std_name,
                             "aliases": new_aliases,
                             "sources": new_sources,
                             "cik": new_cik,
                             "original_indices": new_original_indices} 
        new_row_series = pd.Series(combined_row_dict)
        matched_df = pd.concat([matched_df, new_row_series.to_frame().T])
        
matched_df
          

In [ ]:
# """
# First block the names by the first words of the std_name so that fuzzy matching can reduce the amount of comparisons for performance
# Then match using fuzzy matching and create a list of a list with high scoring matches
# Finally merge into the orginal crosswalk that was matched based on CIK id and std_names
# """
# unmatched_for_fuzzy['first_word'] = unmatched_for_fuzzy['std_name'].fillna('').str.split().str[0]
# unmatched_for_fuzzy.head(20)
# unmatched_for_fuzzy.reset_index(drop = True)

In [ ]:
# from rapidfuzz import process, fuzz

# def fuzzy_matching_df(df_names, threshhold = 90):
#     cols = ['first_word', 'matched_groups']
#     matched_df = pd.DataFrame(columns = cols)
    
#     for name, group in unmatched_for_fuzzy.groupby('first_word'):
#         group_clusters = []
#         candidate_names =group['std_name'].tolist()
        
#         while len(candidate_names) > 0:
#             # picking the main comparison representative based on the longest name
#             representative = max(candidate_names, key = len)
#             current_matches = process.extract(
#                 representative,
#                 candidate_names,
#                 scorer=fuzz.token_set_ratio,
#                 score_cutoff=threshhold,
#                 limit=None                
#             )
#             group_members = [match[0] for match in current_matches]
#             # remove a name from candidate_names if already put in a group
#             candidate_names = [n for n in candidate_names if n not in group_members]
#             group_clusters.append(group_members)
            
#             new_rows = []
#             for cluster in group_clusters:
#                 new_rows.append({
#                     'first_word': name,           
#                     'matched_groups': cluster     # This cluster now holds lists of (std, raw) tuples
#                 })
        
#         matched_df = pd.concat([matched_df, pd.DataFrame(new_rows)], ignore_index=True)
        
#     return matched_df
            
        
        
    

In [ ]:
# ### TO DO: Add raw names to this new data frame

# new_df = fuzzy_matching_df(unmatched_for_fuzzy)
# new_df['raw_names'] = ''
# new_df = new_df.reset_index()
# new_df[new_df['matched_groups'].apply(lambda x: len(x) > 1)]

# new_df.head(100)


In [ ]:
# """
# Merge the matches from fuzzy matching into the previously enriched crosswalk
# First pick a representative from each of the matched groups and then see if any of the std_names match from the enriched crosswalk
#     - Use the blocking method first as well
# If so, merge the entire list into the enriched crosswalk
#     - First get the information about that particular std-name—raw_name, CIK ID, etc 
#     - Merge into the matching row
#     - Add the matching method (fuzzy_matching)
# If the group does not match with any of the std_names, add a new row with all the info about each of the entities in the matched_group
# """

# from tqdm import tqdm

# final_master_crosswalk = final_enriched_crosswalk.copy()
# unmerged_entities = new_df.copy()

# lookup_crosswalk = final_enriched_crosswalk.copy()

# lookup_crosswalk['std_name_single'] = lookup_crosswalk['standardized_names'].str.split('|')
# lookup_crosswalk = lookup_crosswalk.explode('std_name_single')
# lookup_crosswalk = lookup_crosswalk[['std_name_single', 'entity_id']].drop_duplicates()

# lookup_crosswalk['first_word'] = lookup_crosswalk['std_name_single'].fillna('').str.split().str[0]
# unmerged_indices = set()    

# # Wrap groupby iterator with tqdm for a progress bar
# for name, group in tqdm(lookup_crosswalk.groupby('first_word'), desc="Processing first_word groups"):
#     group_clusters = []
#     candidate_names = group['std_name_single'].tolist()
#     target_first_word = group['first_word'].iloc[0]
#     same_first_word_entities = unmerged_entities.loc[unmerged_entities['first_word'] == target_first_word]
#     same_first_word_entities = same_first_word_entities.explode('matched_groups')
    
#     for std_name in same_first_word_entities['matched_groups']:
#         index = same_first_word_entities.loc[same_first_word_entities['matched_groups'] == std_name]['index'].iloc[0]

#         # check if the index associated with this name has already been merged
#         if index in unmerged_indices:
#             continue
         
#         # Individual std_name is in matched_groups now after exploding 
#         best_match = process.extractOne(
#             std_name,
#             candidate_names,
#             scorer=fuzz.token_set_ratio,
#             score_cutoff=90
#         )
        
#         if best_match == None:
#             continue
        
#         matching_entity_id = lookup_crosswalk.loc[lookup_crosswalk['std_name_single'] == best_match[0]]['entity_id'].iloc[0]
#         merging_df = pd.DataFrame()
        
#         names = unmerged_entities.loc[index, 'matched_groups']
#         merging_df = unmatched_for_fuzzy[unmatched_for_fuzzy['std_name'].isin(names)]
        

#         # Get the single row index for this entity_id
#         row_index = final_master_crosswalk.index[
#             final_master_crosswalk['entity_id'] == matching_entity_id
#         ][0]

#         # --- Combine Aliases ---
#         aliases_list = merging_df['raw_name'].dropna().astype(str).tolist()
#         aliases_str = '|'.join(sorted(set(x.strip() for x in aliases_list if x.strip() != '')))
#         final_master_crosswalk.at[row_index, 'aliases'] = combine_pipes(
#             final_master_crosswalk.at[row_index, 'aliases'],
#             aliases_str
#         )

#         # --- Combine Sources ---
#         sources_list = merging_df['source'].dropna().astype(str).tolist()
#         sources_str = ','.join(sorted(set(x.strip() for x in sources_list if x.strip() != '')))
#         final_master_crosswalk.at[row_index, 'sources'] = combine_commas(
#             final_master_crosswalk.at[row_index, 'sources'],
#             sources_str
#         )

#         # --- Combine CIKs (lists) ---
#         final_master_crosswalk.at[row_index, 'cik'] = combine_lists(
#             final_master_crosswalk.at[row_index, 'cik'],
#             merging_df['cik']
#         )

#         # --- Combine Standardized Names ---
#         new_std_names = '|'.join(sorted(set(merging_df['std_name'].dropna().astype(str))))
#         final_master_crosswalk.at[row_index, 'standardized_names'] = combine_pipes(
#             final_master_crosswalk.at[row_index, 'standardized_names'],
#             new_std_names
#         )

#         # --- Update match_type ---
#         final_master_crosswalk.at[row_index, 'match_type'] = combine_commas(
#             final_master_crosswalk.at[row_index, 'match_type'],
#             'fuzzy_matching'
#         )

#         unmerged_indices.add(index)
        

# """
# TO DO:
# 1. if there is a match for that std_name (is not None), then get the index of that row 
# from the same_first_word_entities. Also extract the entity_id for the std_name_single which is matched with.
# 2. For every single name in the matched_groups of that specific column, get the original information from 
# unmatched_for_fuzzy. 
# 3. Use the helper merge functions to merge all that information into the final_master_crosswalk
# 4. remove all the std_names from same_first_word_entities with the same index. 
# 5. After checking all the std_name for all first word groups, then just merge the remaining into the final enriched crosswalk
# """

In [ ]:
# ### TO DO: Increase efficiency for this...

# unmatched_entities = unmerged_entities[~unmerged_entities['index'].isin(unmerged_indices)]

# # Loop over unmatched entities with progress bar
# for idx, row in tqdm(unmatched_entities.iterrows(), total=len(unmatched_entities), desc="Adding unmatched entities"):
#     new_rows = unmatched_for_fuzzy[unmatched_for_fuzzy['std_name'].isin(row['matched_groups'])].copy()
    
#     # Create missing columns for final_master_crosswalk
#     new_rows['match_type'] = 'fuzzy_matching'
#     new_rows['aliases'] = new_rows['raw_name']
#     new_rows['sources'] = new_rows['source']
#     new_rows['entity_id'] = pd.NA  # placeholder if entity_id does not exist
#     new_rows['standardized_names'] = new_rows['std_name']
    
#     new_rows = new_rows[['entity_id', 'aliases', 'sources', 'cik', 'standardized_names', 'match_type']]
    
#     # Append to final_master_crosswalk
#     final_master_crosswalk = pd.concat([final_master_crosswalk, new_rows], ignore_index=True)


In [ ]:
# final_master_crosswalk[final_master_crosswalk['match_type'] == 'fuzzy_matching']

In [ ]:
# filtered_df = final_master_crosswalk[
#     final_master_crosswalk['match_type'].str.contains('fuzzy_matching', na=False)
# ]
# filtered_df